# LAS & Signal Real-World Accuracy Validation

This notebook validates whether the **Lazy Attention Score (LAS)** and the rule-based **buy/sell signals** predict real-world stock returns.

### Why forward returns?

LAS already includes **CAR** (cumulative abnormal return over the filing event window) as 25% of its formula. Measuring LAS accuracy against the same CAR would be circular. Instead we compute **forward abnormal returns** starting **after** the event window ends (day +6 onward, since `CAR_WINDOW = (-1, 5)`).

### Validation methods

| # | Method | What it tells us |
|---|--------|------------------|
| 1 | **Information Coefficient (IC)** | Rank correlation of LAS vs forward returns — the core predictive-power metric |
| 2 | **Quintile Spread** | Do high-LAS filings outperform low-LAS filings? |
| 3 | **Signal Hit Rate** | Are buy/sell signals directionally correct? |
| 4 | **Portfolio Simulation** | What would a signal-following strategy earn? |
| 5 | **Component Attribution** | Which LAS component drives predictive power? |

### Key formula

```
LAS = 0.50 * f(change_intensity) - 0.25 * f(attention_proxy) + 0.25 * f(|CAR|)
```

The **LazyPrices thesis**: material 10-K changes with low investor attention predict post-filing drift over weeks/months.

## Section 1 — Setup and Imports

In [ ]:
import sys, os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config
from store import LASStore
from backtest import (
    load_backtest_data,
    enrich_with_forward_returns,
    assign_signals,
    compute_ic,
    compute_component_ic,
    quintile_analysis,
    signal_hit_rate,
    signal_confusion_matrix,
    portfolio_simulation,
    FORWARD_HORIZONS,
)

HORIZONS = FORWARD_HORIZONS
print(f"Project root: {PROJECT_ROOT}")
print(f"Forward-return horizons (trading days): {HORIZONS}")
print(f"CAR event window: {config.CAR_WINDOW}  \u2192  forward returns start at day +{config.CAR_WINDOW[1]+1}")

## Section 2 — Load Filings and Fetch Forward Returns

Load all scored filings from the database and enrich each with forward abnormal returns at 30, 60, 90, and 180 trading-day horizons. Forward returns are cached to CSV after the first run so subsequent executions skip the Yahoo Finance downloads.

In [ ]:
df = load_backtest_data()
print(f"Loaded {len(df)} scored filings across {df['ticker'].nunique()} tickers")
print(f"Date range: {df['filed_date'].min()} \u2192 {df['filed_date'].max()}")
print(f"\nFilings per ticker:")
print(df.groupby("ticker").size().sort_values(ascending=False).to_string())

In [ ]:
df = enrich_with_forward_returns(df, HORIZONS)
df = assign_signals(df)

fwd_cols = [f"fwd_{h}d" for h in HORIZONS]
print(f"\nForward return coverage:")
for col in fwd_cols:
    n_valid = df[col].notna().sum()
    print(f"  {col}: {n_valid}/{len(df)} filings ({100*n_valid/len(df):.0f}%)")

print(f"\nSignal distribution:")
print(df["signal"].value_counts().to_string())

print(f"\nSample rows:")
display(df[["ticker", "filed_date", "las", "change_intensity", "attention_proxy", "car", "signal"] + fwd_cols].head(10))

## Section 3 — Information Coefficient (IC)

The **Information Coefficient** is the Spearman rank correlation between LAS and forward abnormal returns. We compute it cross-sectionally by year, then average.

- **IC > 0.05** is meaningful for a single quant factor
- **IC > 0.10** is considered strong
- A **positive IC** means higher LAS predicts higher forward returns (consistent with the LazyPrices thesis that material, under-noticed changes lead to drift)

In [ ]:
ic_df = compute_ic(df, HORIZONS)
display(ic_df)

fig, ax = plt.subplots(figsize=(8, 4))
valid_ic = ic_df.dropna(subset=["ic_mean"])
if not valid_ic.empty:
    colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in valid_ic["ic_mean"]]
    bars = ax.bar(valid_ic["horizon"].astype(str) + "d", valid_ic["ic_mean"], color=colors, edgecolor="white", width=0.6)
    if "ic_std" in valid_ic.columns:
        errs = valid_ic["ic_std"].fillna(0)
        ax.errorbar(range(len(valid_ic)), valid_ic["ic_mean"], yerr=errs, fmt="none", color="black", capsize=4)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.axhline(0.05, color="blue", linewidth=0.6, linestyle=":", label="IC = 0.05 benchmark")
    ax.axhline(-0.05, color="blue", linewidth=0.6, linestyle=":")
    ax.set_xlabel("Forward Horizon (trading days)")
    ax.set_ylabel("Mean Spearman IC")
    ax.set_title("Information Coefficient: LAS vs Forward Abnormal Returns")
    ax.legend()
    for bar, val in zip(bars, valid_ic["ic_mean"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
else:
    ax.text(0.5, 0.5, "No IC data available", ha="center", va="center", transform=ax.transAxes)
plt.tight_layout()
plt.show()

## Section 4 — Quintile Spread Analysis

Sort all filings by LAS into 5 equal bins and compare average forward returns across quintiles. The **long-short (L/S) spread** — top quintile minus bottom quintile — is the economic payoff of the LAS signal.

In [ ]:
quint_df = quintile_analysis(df, HORIZONS)
display(quint_df)

quint_data = quint_df[quint_df["quintile"] != "L/S"].copy()
quint_data["quintile"] = quint_data["quintile"].astype(int)

if not quint_data.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    palette = sns.color_palette("viridis", n_colors=len(HORIZONS))

    n_horizons = quint_data["horizon"].nunique()
    n_quintiles = quint_data["quintile"].nunique()
    bar_width = 0.8 / n_horizons
    quintiles = sorted(quint_data["quintile"].unique())

    for i, h in enumerate(sorted(quint_data["horizon"].unique())):
        h_data = quint_data[quint_data["horizon"] == h].set_index("quintile")
        x = np.arange(len(quintiles)) + i * bar_width
        vals = [h_data.loc[q, "mean_return"] * 100 if q in h_data.index else 0 for q in quintiles]
        ax.bar(x, vals, width=bar_width, label=f"{h}d", color=palette[i], edgecolor="white")

    ax.set_xticks(np.arange(len(quintiles)) + bar_width * (n_horizons - 1) / 2)
    ax.set_xticklabels([f"Q{q}\n({'Low' if q == 1 else 'High' if q == max(quintiles) else ''})" for q in quintiles])
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_xlabel("LAS Quintile")
    ax.set_ylabel("Mean Forward Abnormal Return (%)")
    ax.set_title("Quintile Spread: Forward Returns by LAS Rank")
    ax.legend(title="Horizon")
    plt.tight_layout()
    plt.show()

    ls_spread = quint_df[quint_df["quintile"] == "L/S"]
    if not ls_spread.empty:
        print("\nLong/Short Spread (Q5 \u2212 Q1):")
        for _, row in ls_spread.iterrows():
            print(f"  {int(row['horizon'])}d: {row['mean_return']*100:+.2f}%")
else:
    print("Insufficient data for quintile analysis.")

## Section 5 — Signal Hit Rate

For each of the five signal labels (`sell`, `caution`, `hold`, `neutral`, `buy`), we check how often the predicted direction matched the realized forward return.

- **sell/caution**: correct if forward return was negative
- **buy**: correct if forward return was positive
- **hold/neutral**: correct if forward return was within \u00b13% (flat)

In [ ]:
hit_df = signal_hit_rate(df, HORIZONS)
display(hit_df)

if not hit_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Hit rate heatmap
    pivot_hit = hit_df.pivot_table(index="signal", columns="horizon", values="hit_rate")
    signal_order = ["sell", "caution", "hold", "neutral", "buy"]
    signal_order = [s for s in signal_order if s in pivot_hit.index]
    pivot_hit = pivot_hit.reindex(signal_order)

    sns.heatmap(pivot_hit, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1,
                ax=axes[0], linewidths=0.5, cbar_kws={"label": "Hit Rate"})
    axes[0].set_title("Signal Hit Rate by Horizon")
    axes[0].set_ylabel("Signal")
    axes[0].set_xlabel("Forward Horizon (days)")

    # Mean forward return by signal
    pivot_ret = hit_df.pivot_table(index="signal", columns="horizon", values="mean_fwd_return")
    pivot_ret = pivot_ret.reindex(signal_order) * 100

    sns.heatmap(pivot_ret, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
                ax=axes[1], linewidths=0.5, cbar_kws={"label": "Return (%)"})
    axes[1].set_title("Mean Forward Return (%) by Signal")
    axes[1].set_ylabel("Signal")
    axes[1].set_xlabel("Forward Horizon (days)")

    plt.tight_layout()
    plt.show()
else:
    print("No signal hit-rate data available.")

### Signal Confusion Matrix (90-day horizon)

Cross-tabulation of signal vs. realized outcome bucket (negative < \u22122%, flat \u00b12%, positive > +2%) at the 90-day forward horizon.

In [ ]:
confusion = signal_confusion_matrix(df, horizon=90)
if not confusion.empty:
    print("Confusion Matrix \u2014 Signal vs Realized Outcome (90d):\n")
    display(confusion)

    inner = confusion.drop("All", axis=0, errors="ignore").drop("All", axis=1, errors="ignore")
    if not inner.empty:
        fig, ax = plt.subplots(figsize=(7, 4))
        sns.heatmap(inner, annot=True, fmt="d", cmap="Blues", ax=ax, linewidths=0.5)
        ax.set_title("Confusion Matrix: Signal vs 90-day Outcome")
        ax.set_ylabel("Signal")
        ax.set_xlabel("Realized Outcome")
        plt.tight_layout()
        plt.show()
else:
    print("Insufficient data for confusion matrix.")

## Section 6 — Portfolio Simulation (Equity Curve)

Simulate a simple strategy: go **long** on `buy` signals, **short** on `sell` / `caution` signals, and skip `hold` / `neutral`. Each position is held for the given horizon. We compare cumulative returns vs a passive S&P 500 benchmark.

In [ ]:
sim_results = {}
for h in HORIZONS:
    sim_results[h] = portfolio_simulation(df, h)

summary_rows = []
for h, sim in sim_results.items():
    summary_rows.append({
        "Horizon": f"{h}d",
        "Trades": sim.get("n_trades", 0),
        "Mean Return": f"{sim['mean_return']*100:+.2f}%" if sim.get("mean_return") is not None else "N/A",
        "Total Return": f"{sim['total_return']*100:+.2f}%" if sim.get("total_return") is not None else "N/A",
        "Sharpe": f"{sim['sharpe']:.2f}" if sim.get("sharpe") is not None else "N/A",
        "Hit Rate": f"{sim['hit_rate']:.0%}" if sim.get("hit_rate") is not None else "N/A",
        "Max Drawdown": f"{sim['max_drawdown']*100:.1f}%" if sim.get("max_drawdown") is not None else "N/A",
    })

print("Portfolio Simulation Summary:\n")
display(pd.DataFrame(summary_rows))

# Plot equity curves
fig, ax = plt.subplots(figsize=(12, 5))
plotted = False
for h in HORIZONS:
    curve = sim_results[h].get("equity_curve", [])
    if curve:
        dates = [c["date"] for c in curve]
        equity = [c["equity"] for c in curve]
        ax.plot(dates, equity, marker="o", markersize=3, label=f"{h}d horizon")
        plotted = True

if plotted:
    ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--", label="Breakeven")
    ax.set_xlabel("Filing Date")
    ax.set_ylabel("Cumulative Equity ($1 start)")
    ax.set_title("Signal-Based Portfolio: Equity Curve")
    ax.legend()
    tick_spacing = max(1, len(ax.get_xticks()) // 10)
    ax.set_xticks(ax.get_xticks()[::tick_spacing])
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No equity curve data \u2014 no actionable trades found.")

## Section 7 — Component-Level Attribution

Which part of LAS drives the predictive power? We compute IC separately for:

- **change_intensity** — YoY 10-K textual/numerical change
- **attention_proxy** — abnormal volume ratio around filing
- **car** — event-window cumulative abnormal return (circular with LAS)
- **las** — the composite score
- **las_ex_car** — LAS recomputed without the CAR component: `0.67 * change - 0.33 * attention`

In [ ]:
comp_ic = compute_component_ic(df, HORIZONS)
display(comp_ic)

if not comp_ic.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    factors = comp_ic["factor"].unique()
    palette = sns.color_palette("Set2", n_colors=len(factors))
    bar_width = 0.8 / len(factors)

    for i, factor in enumerate(factors):
        f_data = comp_ic[comp_ic["factor"] == factor]
        x = np.arange(len(HORIZONS)) + i * bar_width
        vals = []
        for h in HORIZONS:
            row = f_data[f_data["horizon"] == h]
            vals.append(row["ic"].values[0] if len(row) and row["ic"].values[0] is not None else 0)
        ax.bar(x, vals, width=bar_width, label=factor, color=palette[i], edgecolor="white")

    ax.set_xticks(np.arange(len(HORIZONS)) + bar_width * (len(factors) - 1) / 2)
    ax.set_xticklabels([f"{h}d" for h in HORIZONS])
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Forward Horizon (trading days)")
    ax.set_ylabel("Spearman IC")
    ax.set_title("Component Attribution: IC per Factor vs Forward Returns")
    ax.legend(title="Factor", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No component IC data available.")

## Section 8 — Statistical Summary

Consolidated table of all key validation metrics, plus a scatter plot of LAS vs forward returns.

In [ ]:
print("=" * 70)
print("           LAS & SIGNAL VALIDATION SUMMARY")
print("=" * 70)

print(f"\nUniverse: {df['ticker'].nunique()} tickers, {len(df)} filing observations")
print(f"Date range: {df['filed_date'].min()} to {df['filed_date'].max()}")
print(f"LAS weights: change={config.LAS_WEIGHTS['w_change']}, "
      f"attention={config.LAS_WEIGHTS['w_attention']}, car={config.LAS_WEIGHTS['w_car']}")

print("\n--- Information Coefficient ---")
if not ic_df.empty:
    for _, row in ic_df.iterrows():
        ic_val = row["ic_mean"]
        p_val = row.get("p_value")
        sig = ""
        if p_val is not None and p_val < 0.05:
            sig = " *"
        elif p_val is not None and p_val < 0.10:
            sig = " \u2020"
        print(f"  {int(row['horizon'])}d:  IC = {ic_val:+.4f}{sig}  (n={int(row['n'])})")

print("\n--- Quintile Long/Short Spread ---")
ls = quint_df[quint_df["quintile"] == "L/S"]
if not ls.empty:
    for _, row in ls.iterrows():
        print(f"  {int(row['horizon'])}d:  Q5\u2212Q1 = {row['mean_return']*100:+.2f}%")

print("\n--- Signal Hit Rates (90d) ---")
hr_90 = hit_df[hit_df["horizon"] == 90]
if not hr_90.empty:
    for _, row in hr_90.iterrows():
        print(f"  {row['signal']:>8s}: {row['hit_rate']:.0%}  (n={row['count']}, mean return={row['mean_fwd_return']*100:+.2f}%)")

print("\n--- Portfolio Simulation ---")
for h, sim in sim_results.items():
    if sim.get("n_trades", 0) > 0:
        print(f"  {h}d:  Sharpe={sim.get('sharpe', 'N/A')}, "
              f"Total Return={sim.get('total_return', 0)*100:+.2f}%, "
              f"Hit Rate={sim.get('hit_rate', 0):.0%}, "
              f"Max DD={sim.get('max_drawdown', 0)*100:.1f}%")

print("\n* p < 0.05   \u2020 p < 0.10")

In [ ]:
# Scatter: LAS vs Forward Returns at each horizon
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4), sharey=False)
if len(HORIZONS) == 1:
    axes = [axes]

signal_colors = {"sell": "#e74c3c", "caution": "#e67e22", "hold": "#95a5a6",
                 "neutral": "#3498db", "buy": "#2ecc71"}

for ax, h in zip(axes, HORIZONS):
    col = f"fwd_{h}d"
    sub = df.dropna(subset=[col, "las"]).copy()
    if sub.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        continue

    for sig in ["sell", "caution", "hold", "neutral", "buy"]:
        mask = sub["signal"] == sig
        if mask.any():
            ax.scatter(sub.loc[mask, "las"], sub.loc[mask, col] * 100,
                       c=signal_colors.get(sig, "grey"), label=sig, alpha=0.7, s=40, edgecolors="white")

    slope, intercept, r, p, _ = stats.linregress(sub["las"], sub[col] * 100)
    x_line = np.linspace(sub["las"].min(), sub["las"].max(), 50)
    ax.plot(x_line, intercept + slope * x_line, color="black", linewidth=1.5, linestyle="--",
            label=f"r={r:.2f}, p={p:.3f}")

    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("LAS")
    ax.set_ylabel("Forward Return (%)")
    ax.set_title(f"{h}-day Horizon")
    ax.legend(fontsize=7, loc="best")

plt.suptitle("LAS vs Forward Abnormal Returns (colored by signal)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Section 9 — Caveats and Limitations

### Small sample size
With ~30 DJIA tickers and ~5 filings each, we have roughly 120-150 filing-level observations. Quintile bins contain only ~25-30 observations each. Statistical significance is hard to achieve, and results should be interpreted as directional evidence rather than definitive proof.

### Survivorship bias
The DJIA constituents are the largest, most-covered stocks in the market. The LazyPrices paper finds the strongest effects in **small and mid-cap stocks** where analyst coverage is thin and investor inattention is more common. Our DJIA-focused universe is therefore a conservative test — if we see any signal here, the effect is likely stronger in the broader market.

### Look-ahead in normalization
The current `compute_las()` normalizes `change_intensity`, `attention_proxy`, and `car` via cross-sectional rank within a CIK batch processed together. This is **not** a strict point-in-time normalization. In a production backtest, each filing should be normalized only against data available on its filing date.

### Annual frequency
10-K filings are annual events. With ~5 years of data per ticker, each ticker contributes only a handful of signal events. A robust validation would extend to 10-Q filings (quarterly) and increase the universe beyond the DJIA.

### Transaction costs
The portfolio simulation reports gross returns. Real-world costs — commissions, bid-ask spread, slippage, borrow costs for short positions — would reduce the returns.

### CAR circularity
LAS includes CAR as 25% of its weight. The forward returns used here start **after** the CAR event window ends (day +6), which avoids direct overlap. However, momentum and mean-reversion effects around the filing date could still create indirect correlation. The **ex-CAR LAS** variant in the component attribution section helps quantify this.

### Not investment advice
This analysis is an academic validation exercise for a capstone project. The signals and scores are not financial advice and should not be used for actual trading decisions without further validation, risk management, and regulatory compliance.

In [ ]:
# Save all results to disk
output_dir = os.path.join(config.DATA_DIR, "backtest_results")
os.makedirs(output_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "backtest_data.csv"), index=False)
ic_df.to_csv(os.path.join(output_dir, "ic_results.csv"), index=False)
comp_ic.to_csv(os.path.join(output_dir, "component_ic.csv"), index=False)
quint_df.to_csv(os.path.join(output_dir, "quintile_results.csv"), index=False)
hit_df.to_csv(os.path.join(output_dir, "signal_hit_rates.csv"), index=False)
if not confusion.empty:
    confusion.to_csv(os.path.join(output_dir, "confusion_matrix.csv"))

summary = {
    "n_filings": len(df),
    "n_tickers": int(df["ticker"].nunique()),
    "horizons": HORIZONS,
    "ic": ic_df.to_dict("records"),
    "component_ic": comp_ic.to_dict("records"),
    "quintiles": quint_df.to_dict("records"),
    "signal_hit_rates": hit_df.to_dict("records"),
    "portfolio_simulations": {
        str(k): {kk: vv for kk, vv in v.items() if kk != "equity_curve"}
        for k, v in sim_results.items()
    },
}
with open(os.path.join(output_dir, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"All results saved to {output_dir}/")
print("Files:", os.listdir(output_dir))